<a href="https://colab.research.google.com/github/cikicali369-glitch/DSB-MTN-Git-Starter/blob/main/lab_supervised_learning_model_comparison_Ali_Cikic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/cikicali369-glitch/DSB-Unit-03/blob/main/labs/03%20-%20Supervised-Model-Comparison/assets/ga-logo.png?raw=1" style="float: left; margin: 20px; height: 55px">

# Lab: Supervised Learning Model Comparison
---

Recall the data science process.

1. Define the problem.
2. Gather the data.
3. Explore the data.
4. Model the data.
5. Evaluate the model.
6. Answer the problem.

In this lab, we're going to focus mostly on creating (and then comparing) many regression and classification models. Thus, we'll define the problem and gather the data for you.
Most of the questions requiring a written response can be written in 2-3 sentences.

### Step 1: Define the problem.

You are a data scientist with a financial services company. Specifically, you want to leverage data in order to identify potential customers.

If you are unfamiliar with "401(k)s" or "IRAs," these are two types of retirement accounts. Very broadly speaking:
- You can put money for retirement into both of these accounts.
- The money in these accounts gets invested and hopefully has a lot more money in it when you retire.
- These are a little different from regular bank accounts in that there are certain tax benefits to these accounts. Also, employers frequently match money that you put into a 401k.
- If you want to learn more about them, check out [this site](https://www.nerdwallet.com/article/ira-vs-401k-retirement-accounts).

We will tackle one regression problem and one classification problem today.
- Regression: What features best predict one's income?
- Classification: Predict whether or not one is eligible for a 401k.

Check out the data dictionary [here](http://fmwww.bc.edu/ec-p/data/wooldridge2k/401KSUBS.DES).

### NOTE: When predicting `inc`, you should pretend as though you do not have access to the `e401k`, the `p401k` variable, and the `pira` variable. When predicting `e401k`, you may use the entire dataframe if you wish.

### Step 2: Gather the data.

##### 1. Read in the data from the repository.

In [1]:
import warnings
import numpy as np
import pandas as pd



# Read in the data from the repository.
df = pd.read_csv("https://raw.githubusercontent.com/cikicali369-glitch/DSB-Unit-03/main/labs/03%20-%20Supervised-Model-Comparison/data/401ksubs.csv")
print(df.shape)
df.head()

(9275, 11)


,e401k,inc,marr,male,age,fsize,nettfa,p401k,pira,incsq,agesq
0,0,13.170,0,0,40,1,4.575,0,1,173.4489,1600
1,1,61.230,0,1,35,1,154.000,1,0,3749.1130,1225
2,0,12.858,1,0,44,2,0.000,0,0,165.3282,1936
3,0,98.880,1,1,44,2,21.800,0,0,9777.2540,1936
4,0,22.614,0,0,53,1,18.450,0,0,511.3930,2809


In [2]:
# Check the data types, missing values and the basic statistics.
print(df.isnull().sum().sum(), "missing values")
df.describe()

0 missing values


,e401k,inc,marr,male,age,fsize,nettfa,p401k,pira,incsq,agesq
count,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000,9275.000000
mean,0.392129,39.254641,0.628571,0.204420,41.080216,2.885067,19.071675,0.276226,0.254340,2121.192483,1793.652722
std,0.488252,24.090002,0.483213,0.403299,10.299517,1.525835,63.963838,0.447154,0.435513,3001.469424,895.648841
min,0.000000,10.008000,0.000000,0.000000,25.000000,1.000000,-502.302000,0.000000,0.000000,100.160100,625.000000
25%,0.000000,21.660000,0.000000,0.000000,33.000000,2.000000,-0.500000,0.000000,0.000000,469.155600,1089.000000
50%,0.000000,33.288000,1.000000,0.000000,40.000000,3.000000,2.000000,0.000000,0.000000,1108.091000,1600.000000
75%,1.000000,50.160000,1.000000,0.000000,48.000000,4.000000,18.449500,1.000000,1.000000,2516.025500,2304.000000
max,1.000000,199.041000,1.000000,1.000000,64.000000,13.000000,1536.798000,1.000000,1.000000,39617.320000,4096.000000


##### 2. What are 2-3 other variables that, if available, would be helpful to have?

**Answer:** Education level and job type (occupation) would help, because they strongly affect both income and retirement benefits. A 401(k) is offered by the employer, so the employer type and size (private vs. public sector) and whether the employer offers a match would be very helpful. Also the number of dependents or debts, since they change how much a person can save.

##### 3. Suppose a peer recommended putting `race` into your model in order to better predict who to target when advertising IRAs and 401(k)s. Why would this be an unethical decision?

**Answer:** Using `race` would be discriminatory. It would mean we advertise to people (or ignore people) because of their race and not because of their financial situation, which is unfair and in many places illegal in financial services. It can also repeat old inequalities that are already in the data, so the model would copy the bias.

## Step 3: Explore the data.

##### 4. When attempting to predict income, which feature(s) would we reasonably not use? Why?

**Answer:** We should not use `incsq`, because it is just `inc` squared, so it contains the target and the model would be "cheating". We also should not use `e401k`, `p401k` and `pira` (the lab says to pretend we do not have them), since they are results of income and not something we would know in advance. In the code below we drop all four.

##### 5. What two variables have already been created for us through feature engineering? Come up with a hypothesis as to why subject-matter experts may have done this.
> This need not be a "statistical hypothesis." Just brainstorm why SMEs might have done this!

**Answer:** The two engineered variables are `incsq` (income squared) and `agesq` (age squared). My hypothesis is that experts wanted to capture curved (non-linear) relationships: for example, savings may grow with age and then slow down near retirement, and the effect of income may not be a straight line. A linear model can only fit a line, so the squared columns let it fit a curve.

##### 6. Looking at the data dictionary, one variable description appears to be an error. What is this error, and what do you think the correct value would be?

**Answer:** In the data dictionary the labels of `inc` and `age` are wrong: they say "inc^2" and "age^2", which are the same labels as `incsq` and `agesq`. The correct labels should be `inc` = "annual income, $1000s" and `age` = "age in years" (the data confirms it: `inc` goes from 10 to 199 and `age` from 25 to 64, which are not squared values; the squared columns `incsq` and `agesq` go up to about 39,600 and 4,096).

## Step 4: Model the data. (Part 1: Regression Problem)

Recall:
- Problem: What features best predict one's income?
- When predicting `inc`, you should pretend as though you do not have access to the `e401k`, the `p401k` variable, and the `pira` variable.

##### 7. List all modeling tactics we've learned that could be used to solve a regression problem (as of Wednesday afternoon of Week 6). For each tactic, identify whether it is or is not appropriate for solving this specific regression problem and explain why or why not.

**Answer:** All of these are regression tactics that could be used (the target `inc` is a number):

| Tactic | Appropriate here? | Why |
|---|---|---|
| Multiple linear regression | Yes | Good simple baseline for a numeric target. |
| Ridge / LASSO / ElasticNet | Yes | Same as linear regression but with a penalty against overfitting. We only have 6 features, so it is not really needed. |
| k-nearest neighbors regressor | Yes | Works for numeric targets, but we must scale the data first. |
| Decision tree regressor | Yes | Easy to understand, but it can overfit. |
| Bagged trees | Yes | Many trees on bootstrap samples, less variance than one tree. |
| Random forest | Yes | Like bagging with random features, usually better. |
| AdaBoost regressor | Yes | Many small trees that fix each other's mistakes. |
| Polynomial features | Yes | Already done for us (`agesq`, `incsq`). |
| Logistic regression | No | It is for classification (categories), not for a numeric target. |

##### 8. Regardless of your answer to number 7, fit at least one of each of the following models to attempt to solve the regression problem above:
    - a multiple linear regression model
    - a k-nearest neighbors model
    - a decision tree
    - a set of bagged decision trees
    - a random forest
    - an Adaboost model
    
> As always, be sure to do a train/test split! In order to compare modeling techniques, you should use the same train-test split on each. I recommend setting a random seed here.

> You may find it helpful to set up a pipeline to try each modeling technique, but you are not required to do so!

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, AdaBoostRegressor

# Predict income. Drop the target and the columns we should not use (see question 4).
X = df.drop(columns=['inc', 'e401k', 'p401k', 'pira', 'incsq'])
y = df['inc']

# One train/test split used for ALL models (same random seed).
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(X_train.shape, X_test.shape)

(6956, 6) (2319, 6)


In [4]:
# Put all the regression models in one dictionary.
# Linear regression and kNN need scaled data, so they get a StandardScaler in a pipeline.
reg_models = {
    'Linear regression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'kNN (k=10)': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsRegressor(n_neighbors=10))]),
    'Decision tree': DecisionTreeRegressor(random_state=42),
    'Decision tree (max_depth=4)': DecisionTreeRegressor(max_depth=4, random_state=42),
    'Bagged trees': BaggingRegressor(DecisionTreeRegressor(), n_estimators=100, random_state=42),
    'Random forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42)
}

# Fit every model on the training data.
for name, model in reg_models.items():
    model.fit(X_train, y_train)
    print("Fitted:", name)

Fitted: Linear regression
Fitted: kNN (k=10)
Fitted: Decision tree
Fitted: Decision tree (max_depth=4)
Fitted: Bagged trees
Fitted: Random forest
Fitted: AdaBoost


##### 9. What is bootstrapping?

**Answer:** Bootstrapping means making a new dataset by **sampling from our data with replacement**, with the same size as the original. Because we sample with replacement, some rows appear more than once and some do not appear at all (about 37% are left out). We use it to create many different versions of the training data.

##### 10. What is the difference between a decision tree and a set of bagged decision trees? Be specific and precise!

**Answer:** A single decision tree is fit once on the whole training set, so it is very sensitive to the data (high variance) and tends to overfit. A set of bagged trees fits many trees, each on its own bootstrap sample, and then **averages** their predictions (for classification, they vote). Averaging many trees reduces the variance, so the result is more stable than one tree.

##### 11. What is the difference between a set of bagged decision trees and a random forest? Be specific and precise!

**Answer:** Bagged trees use all the features at every split, so each tree can choose the best feature. A random forest does the same bagging, but at **every split only a random subset of the features** is allowed. This makes the trees different from each other.

##### 12. Why might a random forest be superior to a set of bagged decision trees?
> Hint: Consider the bias-variance tradeoff.

**Answer:** In bagged trees one very strong feature (here `nettfa`) is picked at the top of almost every tree, so the trees look alike (they are correlated) and averaging them reduces the variance less. In a random forest the random feature subsets force the trees to be different, so averaging them reduces the variance more. Each single tree is a bit weaker (a bit more bias), but the whole forest usually has lower total error. In our results the two were almost the same, probably because we only have 6 features.

## Step 5: Evaluate the model. (Part 1: Regression Problem)

##### 13. Using RMSE, evaluate each of the models you fit on both the training and testing data.

In [5]:
from sklearn.metrics import mean_squared_error

# RMSE = square root of the mean squared error, on the training and testing data.
reg_rows = []
for name, model in reg_models.items():
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    reg_rows.append([name, train_rmse, test_rmse, test_rmse - train_rmse])

reg_results = pd.DataFrame(reg_rows, columns=['model', 'train_rmse', 'test_rmse', 'gap'])
reg_results.round(2)

,model,train_rmse,test_rmse,gap
0,Linear regression,20.09,20.98,0.89
1,kNN (k=10),17.73,19.62,1.89
2,Decision tree,2.30,27.45,25.15
3,Decision tree (max_depth=4),18.59,19.32,0.73
4,Bagged trees,7.68,20.50,12.82
5,Random forest,7.67,20.50,12.82
6,AdaBoost,24.30,25.30,1.00


In [6]:
# For comparison: the spread of income itself (an RMSE worse than this is useless).
print(y_train.std())

23.979728874295304


##### 14. Based on training RMSE and testing RMSE, is there evidence of overfitting in any of your models? Which ones?

**Answer:** Yes. The **decision tree** overfits the most: its train RMSE is about 2.3 but its test RMSE is about 27.4. The **bagged trees** and the **random forest** also overfit (train about 7.7, test about 20.5). The kNN model overfits a little (about 17.7 vs 19.6). Linear regression (about 20.1 vs 21.0), the pruned tree (about 18.6 vs 19.3) and AdaBoost (about 24.3 vs 25.3) have small gaps, but AdaBoost has the worst test error, so it underfits.

##### 15. Based on everything we've covered so far, if you had to pick just one model as your final model to use to answer the problem in front of you, which one model would you pick? Defend your choice.

**Answer:** I would pick the **decision tree with `max_depth=4`**. It has one of the lowest test RMSEs (about 19.3), a small gap between train and test (so it does not overfit), and it is simple and easy to explain. The random forest and bagged trees are a bit worse on the test data here (about 20.5) and overfit more. The differences between the best models are small, though.

##### 16. Suppose you wanted to improve the performance of your final model. Brainstorm 2-3 things that, if you had more time, you would attempt.

**Answer:** (1) Use `GridSearchCV` to tune the hyperparameters (for example `max_depth`, `min_samples_leaf` for the trees, `n_neighbors` for kNN). (2) Add more useful features, such as education or job type, or interactions like `age * nettfa`. (3) Try a log transformation of `nettfa` and `inc`, because they are very skewed, and check outliers.

## Step 4: Model the data. (Part 2: Classification Problem)

Recall:
- Problem: Predict whether or not one is eligible for a 401k.
- When predicting `e401k`, you may use the entire dataframe if you wish.

##### 17. While you're allowed to use every variable in your dataframe, mention at least one disadvantage of using `p401k` in your model.

In [7]:
# Is anyone who participates in a 401(k) NOT eligible?
pd.crosstab(df['p401k'], df['e401k'])

e401k,0,1
p401k,,
0,5638,1075
1,0,2562


**Answer:** The table shows that everybody who participates (`p401k = 1`) is also eligible (`e401k = 1`). So `p401k` almost gives away the answer (data leakage). A disadvantage is that the model will look much better than it really is, and for a new customer we would not know `p401k` in advance. In my results the logistic regression drops from an f1 of about 0.83 to about 0.48 when `p401k` is removed (see question 27).

##### 18. List all modeling tactics we've learned that could be used to solve a classification problem (as of Wednesday afternoon of Week 6). For each tactic, identify whether it is or is not appropriate for solving this specific classification problem and explain why or why not.

**Answer:** For a classification problem (the target is 0/1) we can use:

| Tactic | Appropriate here? | Why |
|---|---|---|
| Logistic regression | Yes | Made for a yes/no target; also with Ridge/LASSO penalty. |
| k-nearest neighbors classifier | Yes | Works for classes, needs scaled data. |
| Decision tree classifier | Yes | Easy to understand, can overfit. |
| Bagged trees | Yes | Many trees that vote, less variance. |
| Random forest | Yes | Bagging with random features. |
| AdaBoost classifier | Yes | Boosting of small trees. |
| Linear regression / regression versions of the models | No | They predict numbers, not classes. |

##### 19. Regardless of your answer to number 18, fit at least one of each of the following models to attempt to solve the classification problem above:
    - a logistic regression model
    - a k-nearest neighbors model
    - a decision tree
    - a set of bagged decision trees
    - a random forest
    - an Adaboost model
    
> As always, be sure to do a train/test split! In order to compare modeling techniques, you should use the same train-test split on each. I recommend using a random seed here.

> You may find it helpful to set up a pipeline to try each modeling technique, but you are not required to do so!

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier

# Predict e401k. The lab says we may use all other columns.
Xc = df.drop(columns='e401k')
yc = df['e401k']

# One train/test split used for ALL models. stratify keeps the same share of 1s in both parts.
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, random_state=42, stratify=yc)

# Baseline: always guess the most common class.
print(yc.value_counts(normalize=True))

e401k
0    0.607871
1    0.392129
Name: proportion, dtype: float64


In [9]:
clf_models = {
    'Logistic regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression())]),
    'kNN (k=10)': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=10))]),
    'Decision tree': DecisionTreeClassifier(random_state=42),
    'Bagged trees': BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=42),
    'Random forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42)
}

# Fit every model on the training data.
for name, model in clf_models.items():
    model.fit(Xc_train, yc_train)
    print("Fitted:", name)

Fitted: Logistic regression
Fitted: kNN (k=10)
Fitted: Decision tree
Fitted: Bagged trees
Fitted: Random forest
Fitted: AdaBoost


## Step 5: Evaluate the model. (Part 2: Classfication Problem)

##### 20. Suppose our "positive" class is that someone is eligible for a 401(k). What are our false positives? What are our false negatives?

**Answer:** A **false positive** is a person we predict to be eligible for a 401(k), but who is actually **not** eligible. A **false negative** is a person we predict to be **not** eligible, but who actually **is** eligible.

##### 21. In this specific case, would we rather minimize false positives or minimize false negatives? Defend your choice.

**Answer:** I would rather minimize **false negatives**. Our goal is to find potential customers, so a false negative means we miss a real customer and lose business. A false positive only means we send an ad to someone who cannot use it, which costs little.

##### 22. Suppose we wanted to optimize for the answer you provided in problem 21. Which metric would we optimize in this case?

**Answer:** We would optimize **recall** (also called sensitivity) = TP / (TP + FN). It goes up when the number of false negatives goes down.

##### 23. Suppose that instead of optimizing for the metric in problem 21, we wanted to balance our false positives and false negatives using `f1-score`. Why might [f1-score](https://en.wikipedia.org/wiki/F1_score) be an appropriate metric to use here?

**Answer:** The f1-score is the harmonic mean of precision and recall: F1 = 2 * (precision * recall) / (precision + recall). The harmonic mean is pulled toward the lower value, so F1 is only high when **both** precision (few false positives) and recall (few false negatives) are good. That is why it balances false positives and false negatives. It also focuses on the positive class (eligible people) and does not let the many true negatives make the score look good, which can happen with accuracy (our classes are 39% eligible vs 61% not eligible).

##### 24. Using f1-score, evaluate each of the models you fit on both the training and testing data.

In [10]:
from sklearn.metrics import f1_score

# f1-score on the training and testing data (the positive class is 1 = eligible).
clf_rows = []
for name, model in clf_models.items():
    train_f1 = f1_score(yc_train, model.predict(Xc_train))
    test_f1 = f1_score(yc_test, model.predict(Xc_test))
    clf_rows.append([name, train_f1, test_f1, train_f1 - test_f1])

clf_results = pd.DataFrame(clf_rows, columns=['model', 'train_f1', 'test_f1', 'gap'])
clf_results.round(3)

,model,train_f1,test_f1,gap
0,Logistic regression,0.825,0.830,-0.005
1,kNN (k=10),0.828,0.821,0.007
2,Decision tree,1.000,0.755,0.245
3,Bagged trees,1.000,0.823,0.177
4,Random forest,1.000,0.826,0.174
5,AdaBoost,0.825,0.830,-0.005


##### 25. Based on training f1-score and testing f1-score, is there evidence of overfitting in any of your models? Which ones?

**Answer:** Yes. The **decision tree** overfits the most (train f1 = 1.0, test about 0.76). The **bagged trees** and the **random forest** also overfit (train 1.0, test about 0.82). Logistic regression, kNN and AdaBoost have almost the same train and test f1 (about 0.82-0.83), so they do not overfit.

##### 26. Based on everything we've covered so far, if you had to pick just one model as your final model to use to answer the problem in front of you, which one model would you pick? Defend your choice.

**Answer:** I would pick **logistic regression**. Its test f1 (about 0.83) is the best, tied with AdaBoost, and the train and test scores are almost equal, so it does not overfit. It is also the simplest model and its coefficients are easy to interpret. The limitation is that this result is mostly because we used `p401k` (see question 17).

##### 27. Suppose you wanted to improve the performance of your final model. Brainstorm 2-3 things that, if you had more time, you would attempt.

In [11]:
# Check: how good is logistic regression without p401k?
no_p = Xc_train.drop(columns='p401k')
model = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression())])
model.fit(no_p, yc_train)
print(f1_score(yc_train, model.predict(no_p)))
print(f1_score(yc_test, model.predict(Xc_test.drop(columns='p401k'))))

0.46811337466784764
0.484


**Answer:** (1) Tune the hyperparameters with `GridSearchCV` and use the f1-score or recall as the scoring metric. (2) Lower the probability threshold below 0.5 to catch more eligible people (higher recall, fewer false negatives), since that is what we wanted in question 21. (3) Drop `p401k` and add better features (job type, employer size, education), because without `p401k` the f1 is only about 0.48.

## Step 6: Answer the problem.

##### BONUS: Briefly summarize your answers to the regression and classification problems. Be sure to include any limitations or hesitations in your answer.

- Regression: What features best predict one's income?
- Classification: Predict whether or not one is eligible for a 401k.

In [12]:
# Which features matter most for predicting income? (from the pruned decision tree)
tree = reg_models['Decision tree (max_depth=4)']
pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False).round(3)

,0
nettfa,0.721
marr,0.253
age,0.018
agesq,0.008
male,0.000
fsize,0.000


**Regression:** The feature that best predicts income is **`nettfa`** (net financial assets), followed by **`marr`** (married people tend to have a higher household income). Age and family size matter less. My best model (the pruned tree) has a test RMSE of about 19.3 (in $1000s), while the spread of income itself is about 24, so it explains some of the income but is **not very accurate**. A limitation is that we have only 6 features, and I had to leave out `incsq` because it contains the answer.

**Classification:** I can predict 401(k) eligibility with an f1-score of about 0.83 using logistic regression (the baseline of always guessing "not eligible" gives 61% accuracy). However, this is mostly thanks to `p401k`, which is not available for new customers. Without it the f1 is only about 0.48, so the model is weak in a real situation. We should be careful about using these models to make decisions about people.